# Load Libs

In [ ]:
pip install --upgrade onnx onnxscript


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
import os 
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import onnx

# Shaping tha Dataset

In [ ]:
class ShapesDataset(Dataset):

    
    def __init__(self, csv_file, img_dir, transform=None):
        self.labels_df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.class_folders = ['class0', 'class1', 'class2']
        self.class_names = ['Square', 'Triangle', 'Circle']
        
        print(f"csv shape: {self.labels_df.shape}")
        print(f"csv columns: {self.labels_df.columns.tolist()}")


    
    def __len__(self):
        return len(self.labels_df)



    
    def __getitem__(self, idx):
        img_name = str(self.labels_df.iloc[idx, 0])
        label = int(self.labels_df.iloc[idx, 1])
        
        folder_name = self.class_folders[label]
        img_path = os.path.join(self.img_dir, folder_name, img_name)
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Transform and import data 

In [ ]:
from torchvision.datasets import ImageFolder



transform = transforms.Compose([transforms.Resize((32, 32)),
                                transforms.RandomHorizontalFlip(p=0.3),
                                transforms.RandomRotation(10),
                                transforms.ToTensor(),
                                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])




dataset = ImageFolder(root="/kaggle/input/datasets/carlodemello/shapes2/Shapes",  # so this points to the folder containing class0, class1, class2
                      transform=transform)




print(f"total images: {len(dataset)}")
print(f"classes: {dataset.classes}")
print(f"class to index: {dataset.class_to_idx}")

# Labeling the data

In [ ]:
from collections import Counter
labels = [label for _, label in dataset]


print(f"class distribution: {Counter(labels)}")

# Splitting data 

In [ ]:
from torch.utils.data import random_split

train_size = 18002
test_size = len(dataset) - train_size   # here I used the ratio of 60 - 40, you can use 70 -30 or 80 -20, so a lot of r&d

if test_size < 0:
    test_size = 0
    train_size = len(dataset)

print(f"training Data: {train_size}")
print(f"test Data: {test_size}")

train_set, test_set = random_split(dataset, [train_size, test_size])

# giving batch size 

In [ ]:
batch_size = 64
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)




print(f"train batches: {len(train_loader)}")
print(f"test batches: {len(test_loader)}")





images, labels = next(iter(train_loader))       # so this shows you the sample
print(f"batch shape: {images.shape}")
print(f"labels: {labels[:5]}")

# Loading CNN model

In [ ]:
class ShapesCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(nn.Conv2d(3, 32, kernel_size=3, padding=1),
                           nn.BatchNorm2d(32),
                           nn.ReLU(),
                           nn.MaxPool2d(2),
            
                           nn.Conv2d(32, 64, kernel_size=3, padding=1),
                           nn.BatchNorm2d(64),
                           nn.ReLU(),
                           nn.MaxPool2d(2),
            
                           nn.Conv2d(64, 128, kernel_size=3, padding=1),
                           nn.BatchNorm2d(128),
                           nn.ReLU(),
                           nn.MaxPool2d(2))
        
        self.classifier = nn.Sequential(nn.Flatten(),
                          nn.Linear(128 * 4 * 4, 256),
                          nn.ReLU(),
                          nn.Dropout(0.5),
                          nn.Linear(256, 128),
                          nn.ReLU(),
                          nn.Dropout(0.3),
                          nn.Linear(128, 3))
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
                    
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.classifier(x)
        return x

# Setting CNN model 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ShapesCNN().to(device)



learning_rate = 0.0005
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)



print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Assigning loader, model, loss function and optimizer

In [ ]:
def train_epoch(loader, model, loss_fn, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    train_acc = 100. * correct / total
    avg_loss = total_loss / len(loader)
    return avg_loss, train_acc

In [ ]:
def validate_epoch(loader, model, loss_fn):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    
    val_acc = 100. * correct / total
    avg_loss = total_loss / len(loader)
    return avg_loss, val_acc

# starting model training with given epochs as per trial basis

In [ ]:
epochs = 5
best_acc = 0
patience = 5
patience_counter = 0



train_losses = []
train_accs = []
val_losses = []
val_accs = []




for epoch in range(epochs):
    print(f'\nEpoch {epoch+1}/{epochs}')
    
    train_loss, train_acc = train_epoch(train_loader, model, loss_fn, optimizer)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    val_loss, val_acc = validate_epoch(test_loader, model, loss_fn)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    print(f'train loss: {train_loss:.4f}, train acc: {train_acc:.2f}%')
    print(f'val loss: {val_loss:.4f}, val acc: {val_acc:.2f}%')
    
    scheduler.step(val_loss)



    
    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'new best model saved: {val_acc:.2f}% ***')

    
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'  early stopping after {epoch+1} epochs')
            break

In [ ]:
model.load_state_dict(torch.load('best_model.pth', weights_only=True))
model.eval()

In [ ]:
all_preds = []
all_labels = []
all_probs = []



with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        plt.imshow(outputs)
        import matplotlib.pyplot as plt
        probs = torch.nn.functional.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())


all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)



accuracy = np.mean(all_preds == all_labels) * 100
print(f"\nfinal test accuracy: {accuracy:.2f}%")

# Classification report 

# load libs

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
import os 
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import time
from torchvision.datasets import ImageFolder

# transform and load data

In [ ]:
transform = transforms.Compose([transforms.Resize((32, 32)),
                                transforms.ToTensor(),
                                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])



dataset = ImageFolder(root="/kaggle/input/datasets/carlodemello/shapes2/Shapes",
                      transform=transform)



print(f"total images: {len(dataset)}")
print(f"classes: {dataset.classes}")
print(f"class to index: {dataset.class_to_idx}")





train_size = 18002
test_size = len(dataset) - train_size




train_set, test_set = random_split(dataset, [train_size, test_size])




batch_size = 64
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)




print(f"\ntrain batches: {len(train_loader)}")
print(f"test batches: {len(test_loader)}")

In [ ]:
conv_layers_data = {i: [] for i in range(1, 13)}



In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(nn.Conv2d(3, 32, kernel_size=3, padding=1),
                                         nn.BatchNorm2d(32),
                                         nn.ReLU(),
                                         nn.MaxPool2d(2),
            
                                         nn.Conv2d(32, 64, kernel_size=3, padding=1),
                                         nn.BatchNorm2d(64),
                                         nn.ReLU(),
                                         nn.MaxPool2d(2),
            
                                         nn.Conv2d(64, 128, kernel_size=3, padding=1),
                                         nn.BatchNorm2d(128),
                                         nn.ReLU(),
                                         nn.MaxPool2d(2),
            
                                         nn.Conv2d(128, 256, kernel_size=3, padding=1),
                                         nn.BatchNorm2d(256),
                                         nn.ReLU(),
                                         nn.MaxPool2d(2))

        
        
        self.classifier = nn.Sequential(nn.Flatten(),
                                        nn.Linear(256 * 2 * 2, 512),
                                        nn.ReLU(),
                                        nn.Dropout(0.5),
                                        nn.Linear(512, 256),
                                        nn.ReLU(),
                                        nn.Dropout(0.3),
                                        nn.Linear(256, 3))
        def forward(self, x):
            x = self.conv_layers(x)
            x = self.classifier(x)
            return x

In [ ]:
conv_layers_data

# training loop

In [ ]:
def train_model(model, train_loader, test_loader, model_name, epochs=15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"training {model_name}")



    
    model = model.to(device)
    
    learning_rate = 0.001
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)


    
    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []
    training_time = 0


    
    best_acc = 0
    best_model_state = None


    
    for epoch in range(epochs):
        start_time = time.time()
        
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0



        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
        
        epoch_time = time.time() - start_time
        training_time += epoch_time
        
        train_loss = train_loss / len(train_loader)
        train_acc = 100. * train_correct / train_total
        
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0



        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = loss_fn(outputs, labels)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        val_loss = val_loss / len(test_loader)
        val_acc = 100. * val_correct / val_total
        
        scheduler.step()
        
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)




        
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_state = model.state_dict().copy()
        
        print(f'epoch {epoch+1}/{epochs} | time: {epoch_time:.1f}s')
        print(f'  train Loss: {train_loss:.4f} | train Acc: {train_acc:.2f}%')
        print(f'  val Loss:  {val_loss:.4f} | val Acc:  {val_acc:.2f}%')




    
    model.load_state_dict(best_model_state)
    
    torch.save(model.state_dict(), f'{model_name}_best.pth')
    
    results = {'model_name': model_name,
               'train_losses': train_losses,
               'train_accs': train_accs,
               'val_losses': val_losses,
               'val_accs': val_accs,
               'best_val_acc': best_acc,
               'training_time': training_time,
               'model': model,
               'params': sum(p.numel() for p in model.parameters() if p.requires_grad)}



    
    
    print(f"\nbest validation accuracy: {best_acc:.2f}%")
    print(f"total training time: {training_time:.1f}s")
    print(f"trainable parameters: {results['params']:,}")
    
    return results

# testing loop

In [ ]:
def evaluate_model(model, test_loader, model_name):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()


    
    all_preds = []
    all_labels = []
    all_probs = []



    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)



    
    accuracy = np.mean(all_preds == all_labels) * 100
    
    correct_mask = all_preds == all_labels
    correct_confidences = np.array(all_probs)[correct_mask].max(axis=1)


    
    evaluation = {'accuracy': accuracy,
                  'predictions': all_preds,
                  'labels': all_labels,
                  'probs': all_probs,
                  'avg_confidence': correct_confidences.mean() if len(correct_confidences) > 0 else 0}

    
    return evaluation

# plotting the model

# Run CNN model

In [ ]:
cnn_model = CNNModel()
cnn_results = train_model(cnn_model, train_loader, test_loader, "CNN", epochs=5)
plot_training_history(cnn_results)

In [ ]:
torch_model = CNNModel()
# Create example inputs for exporting the model. The inputs should be a tuple of tensors.
example_inputs = (torch.randn(3, 3, 32, 32),)
onnx_program = torch.onnx.export(torch_model, example_inputs, dynamo=True)
onnx_program.save("image_classifier_model.onnx")

In [ ]:

for i in model.conv_layers:
    print(i)

In [ ]:
print(conv_layers_data)

In [ ]:
print(len(model.conv_layers))